# 185. 把 Agent Loop 建模为 POMDP：Belief State、主动观察与安全行动怎样实现？

> **面试问题：为什么 observation 不等于真实 state？Agent 怎样用 noisy tool result 更新 belief，并在高风险动作前主动取证？**

## 先给结论

真实网页、GUI 和工具只暴露部分且可能过期的 observation。POMDP 视角让 Agent 维护对隐藏 state 的 belief，经 observation likelihood 做 Bayesian update，再按期望效用和信息价值选 action。高风险动作不应由单次模型猜测触发；应设置信念阈值、主动观察、状态新鲜度和执行后验证。

## 推荐回答主线

1. 定义隐藏 state、observation、action、transition/reward 与 belief，区分原始历史和充分统计量。
2. 实现 Bayesian belief update，验证归一化、证据顺序和不可能 observation 的失败策略。
3. 比较 inspect/execute/abort 的期望效用与信息增益，高风险动作要求低危险概率和新鲜证据。
4. 把 belief/observation 版本写入 trace；用成功、损害、观察成本、校准和恢复能力评估。

## 教学实现边界

二状态玩具环境的 likelihood 已知；真实 Agent 的状态空间和 observation model 通常未知，只能近似。公式用于组织安全决策，不声称 LLM 能精确做 Bayesian inference。

## 一手资料

- [OSWorld](https://arxiv.org/abs/2404.07972)
- [WebArena](https://arxiv.org/abs/2307.13854)
- [AgentBench](https://arxiv.org/abs/2308.03688)


In [ ]:
import hashlib
import json
import math
from dataclasses import asdict, dataclass

import numpy as np

# 隐藏状态 SAFE/DANGER；工具只返回带版本和时间的有噪声 observation。
STATES = ("safe", "danger")
OBSERVATIONS = ("green", "red")
LIKELIHOOD = {
    "safe": {"green": 0.9, "red": 0.1},
    "danger": {"green": 0.2, "red": 0.8},
}

@dataclass(frozen=True)
class ObservationRecord:
    value: str
    source: str
    state_version: int
    observed_at: int

@dataclass(frozen=True)
class POMDPArtifact:
    environment: str
    observation_model: str
    action_policy: str
    danger_threshold: float
    freshness_limit: int
    inspect_cost: float

belief = np.array([0.6, 0.4], dtype=float)

assert math.isclose(belief.sum(), 1.0)
assert set(LIKELIHOOD) == set(STATES)
assert all(math.isclose(sum(row.values()), 1.0) for row in LIKELIHOOD.values())


## 1. Bayesian update：Observation 是证据，不是 state 标签

posterior 与 prior×likelihood 成正比。一次 green 不能把 danger 概率设成 0；工具也可能错。归一化常数为零时应拒绝更新并记录 model mismatch。


In [ ]:
def validate_belief(prior):
    array = np.asarray(prior, dtype=float)
    if array.shape != (len(STATES),) or not np.isfinite(array).all() or (array < 0).any():
        raise ValueError("prior 必须是有限、非负且与状态数一致的向量")
    if not math.isclose(float(array.sum()), 1.0, rel_tol=0.0, abs_tol=1e-9):
        raise ValueError("prior 概率和必须为 1")
    return array.copy()

def validate_likelihood(likelihood):
    if set(likelihood) != set(STATES):
        raise ValueError("likelihood 状态集合错误")
    for state in STATES:
        row = likelihood[state]
        if set(row) != set(OBSERVATIONS):
            raise ValueError("likelihood observation 集合错误")
        values = np.array([row[item] for item in OBSERVATIONS], dtype=float)
        if not np.isfinite(values).all() or (values < 0).any() or not math.isclose(float(values.sum()), 1.0, abs_tol=1e-9):
            raise ValueError("likelihood 每行必须是有限概率分布")
    return likelihood

def validate_observation_record(record):
    if not isinstance(record, ObservationRecord):
        raise TypeError("observation_stream 只能产生 ObservationRecord")
    if record.value not in OBSERVATIONS or not isinstance(record.source, str) or not record.source:
        raise ValueError("observation value/source 非法")
    if type(record.state_version) is not int or type(record.observed_at) is not int or min(record.state_version, record.observed_at) < 0:
        raise ValueError("observation version/time 必须是非负整数")
    return record

def observation_status(record, current_version, now, max_age):
    validate_observation_record(record)
    if any(type(value) is not int for value in (current_version, now, max_age)) or min(current_version, now, max_age) < 0:
        raise ValueError("current_version/now/max_age 必须是非负整数")
    if record.state_version != current_version:
        return "wrong_version"
    age = now - record.observed_at
    if age < 0:
        return "future"
    if age > max_age:
        return "stale"
    return "accepted"

def version_matches(record, current_version, now, max_age):
    return observation_status(record, current_version, now, max_age) == "accepted"

def update_belief(prior, observation, likelihood=LIKELIHOOD):
    prior = validate_belief(prior)
    validate_likelihood(likelihood)
    if observation not in OBSERVATIONS:
        raise ValueError("未知 observation")
    evidence = np.array([likelihood[state][observation] for state in STATES], dtype=float)
    unnormalized = prior * evidence
    if not np.isfinite(unnormalized).all() or unnormalized.sum() <= 0:
        raise ValueError("observation 在模型下不可能")
    return unnormalized / unnormalized.sum()

# green/red 方向正确；非法 prior 与未知 observation 均 fail closed。
after_green = update_belief(belief, "green")
after_red = update_belief(belief, "red")
assert 0 < after_green[1] < belief[1] < after_red[1]
assert math.isclose(after_green.sum(), 1.0)
for invalid_prior in (np.array([float("nan"), 0.0]), np.array([-0.1, 1.1]), np.array([0.2, 0.2])):
    try:
        update_belief(invalid_prior, "green"); assert False
    except ValueError:
        assert True


## 2. 连续观察：条件独立假设若不成立会过度自信

在给定 state 条件独立的简化下可顺序更新；若两次结果来自同一缓存/同一模型，它们高度相关，不能当独立证据重复乘 likelihood。provenance 要标 observation source 与 freshness。


In [ ]:
def apply_observations(prior, observations, likelihood=LIKELIHOOD):
    posterior = validate_belief(prior)
    for observation in observations:
        posterior = update_belief(posterior, observation, likelihood)
    return posterior

# 独立观测可顺序更新；未知观测不能被当作无害空值跳过。
two_green = apply_observations(belief, ["green", "green"])
mixed_a = apply_observations(belief, ["green", "red"])
mixed_b = apply_observations(belief, ["red", "green"])
assert two_green[1] < after_green[1]
assert np.allclose(mixed_a, mixed_b)
try:
    apply_observations(belief, ["blue"]); assert False
except ValueError:
    assert True


## 3. 期望效用：execute 的收益与损害不对称

安全状态执行收益 +10，危险状态执行损失 -50，abort 为 0。即使 safe 概率过半，期望效用仍可能为负。风险偏好和不可逆损害应由产品 policy 设定，不交给 prompt 临时决定。


In [ ]:
UTILITY = {"execute": np.array([10.0, -50.0]), "abort": np.array([0.0, 0.0])}

def expected_utility(action, current_belief):
    if action not in UTILITY:
        raise ValueError("未知 action")
    return float(validate_belief(current_belief) @ UTILITY[action])

# 初始执行期望为负；green 后改善；abort 恒为零。
assert expected_utility("execute", belief) < 0
assert expected_utility("execute", after_green) > expected_utility("execute", belief)
assert expected_utility("abort", after_red) == 0


## 4. 信息增益：Inspect 的价值是改变后续决策，不是多调用工具

belief entropy 衡量不确定性。inspect 的期望信息增益是观察前熵减去各 observation posterior 熵的期望，再减调用成本。若无论观察什么都不会改变动作，继续 inspect 可能浪费预算。


In [ ]:
def belief_entropy(probabilities):
    p = validate_belief(probabilities)
    positive = p > 0
    return float(-(p[positive] * np.log(p[positive])).sum())

def expected_information_gain(prior, likelihood=LIKELIHOOD):
    prior = validate_belief(prior)
    validate_likelihood(likelihood)
    before, expected_after = belief_entropy(prior), 0.0
    for observation in OBSERVATIONS:
        p_observation = sum(prior[index] * likelihood[state][observation] for index, state in enumerate(STATES))
        if p_observation > 0:
            expected_after += p_observation * belief_entropy(update_belief(prior, observation, likelihood))
    return max(0.0, before - expected_after)

UNINFORMATIVE_LIKELIHOOD = {
    "safe": {"green": 0.5, "red": 0.5},
    "danger": {"green": 0.5, "red": 0.5},
}

# 有信息传感器 EIG>0；无信息传感器和确定 belief 的 EIG 为零。
information_gain = expected_information_gain(belief)
assert information_gain > 0
assert expected_information_gain(belief, UNINFORMATIVE_LIKELIHOOD) < 1e-12
assert expected_information_gain(np.array([1.0, 0.0])) < 1e-12


## 5. 主动观察策略：不确定或高风险时先 inspect

可设 danger 上限与 freshness 门禁：只有新鲜 posterior 且危险概率低于阈值才 execute；否则在预算内 inspect，耗尽后 abort/升级人工。阈值来自损害成本和校准，不是 0.5。


In [ ]:
def validate_policy(policy):
    if not isinstance(policy, POMDPArtifact):
        raise TypeError("policy 必须是 POMDPArtifact")
    if not all(isinstance(value, str) and value for value in (policy.environment, policy.observation_model, policy.action_policy)):
        raise ValueError("artifact 版本字段不得为空")
    if not isinstance(policy.danger_threshold, (int, float)) or not math.isfinite(policy.danger_threshold) or not 0 <= policy.danger_threshold <= 1:
        raise ValueError("danger_threshold 必须在 [0,1]")
    if type(policy.freshness_limit) is not int or policy.freshness_limit < 0:
        raise ValueError("freshness_limit 必须是非负整数")
    if not isinstance(policy.inspect_cost, (int, float)) or not math.isfinite(policy.inspect_cost) or policy.inspect_cost < 0:
        raise ValueError("inspect_cost 必须有限非负")
    return policy

DEFAULT_POLICY = POMDPArtifact("env-v5", "sensor-likelihood-v3", "evidence-gated-eig-v3", 0.08, 2, 0.01)

def choose_action(current_belief, evidence_count, observation_age, inspect_budget, policy=DEFAULT_POLICY, likelihood=LIKELIHOOD, return_diagnostics=False):
    current_belief = validate_belief(current_belief)
    validate_policy(policy); validate_likelihood(likelihood)
    if type(evidence_count) is not int or evidence_count < 0 or type(inspect_budget) is not int or inspect_budget < 0:
        raise ValueError("evidence_count/inspect_budget 必须是非负整数")
    if evidence_count == 0:
        if observation_age is not None:
            raise ValueError("无有效证据时 observation_age 必须为 None")
        fresh_evidence = False
    else:
        if type(observation_age) is not int or observation_age < 0:
            raise ValueError("observation_age 必须是非负整数")
        fresh_evidence = observation_age <= policy.freshness_limit
    danger = float(current_belief[1])
    eig = expected_information_gain(current_belief, likelihood)
    net_information_value = eig - policy.inspect_cost
    can_execute = fresh_evidence and danger <= policy.danger_threshold and expected_utility("execute", current_belief) > 0
    action = "execute" if can_execute else "inspect" if inspect_budget > 0 and net_information_value > 0 else "abort"
    diagnostics = {"eig": eig, "inspect_cost": policy.inspect_cost, "net_information_value": net_information_value, "fresh_evidence": fresh_evidence}
    return (action, diagnostics) if return_diagnostics else action

# 无证据不能 execute；EIG 扣成本后决定是否 inspect；无信息传感器不浪费预算。
assert choose_action(belief, 0, None, 2) == "inspect"
assert choose_action(two_green, 2, 0, 1) == "execute"
assert choose_action(np.array([1.0, 0.0]), 0, None, 2) == "abort"
assert choose_action(belief, 0, None, 2, likelihood=UNINFORMATIVE_LIKELIHOOD) == "abort"
for bad_budget in (-1, 1.5):
    try:
        choose_action(belief, 0, None, bad_budget); assert False
    except ValueError:
        assert True


## 6. Belief-state Agent Loop：Observe→Update→Decide→Act/Stop

循环记录每次 observation 的 source/version/time，更新 belief 后再决策。执行前可做 compare-and-swap/页面重截图，执行后验证状态；这里用预设观测序列演示有界终止。


In [ ]:
def run_belief_agent(prior, observation_stream, inspect_budget=3, current_version=0, now=0, policy=DEFAULT_POLICY, likelihood=LIKELIHOOD):
    posterior = validate_belief(prior)
    validate_policy(policy); validate_likelihood(likelihood)
    if type(inspect_budget) is not int or inspect_budget < 0:
        raise ValueError("inspect_budget 必须是非负整数")
    if any(type(value) is not int for value in (current_version, now)) or min(current_version, now) < 0:
        raise ValueError("current_version/now 必须是非负整数")
    try:
        stream = iter(observation_stream)
    except TypeError as error:
        raise TypeError("observation_stream 必须可迭代") from error
    trace, evidence_count, observation_age = [], 0, None
    while True:
        action, diagnostics = choose_action(posterior, evidence_count, observation_age, inspect_budget, policy, likelihood, True)
        trace.append({"state": "decide", "belief": posterior.copy(), "action": action, "evidence_count": evidence_count, "observation_age": observation_age, **diagnostics})
        if action in {"execute", "abort"}:
            return action, posterior, trace
        try:
            record = next(stream)
        except StopIteration:
            trace.append({"state": "inspect", "status": "stream_exhausted"})
            return "abort", posterior, trace
        inspect_budget -= 1
        status = observation_status(record, current_version, now, policy.freshness_limit)
        trace.append({"state": "observe", "value": record.value, "source": record.source, "state_version": record.state_version, "observed_at": record.observed_at, "status": status})
        if status != "accepted":
            continue
        posterior = update_belief(posterior, record.value, likelihood)
        evidence_count += 1
        observation_age = now - record.observed_at

# 两条新鲜 green 经真实 loop 后执行；red 且预算耗尽则 abort。
good_records = [ObservationRecord("green", "sensor-a", 7, 100), ObservationRecord("green", "sensor-b", 7, 100)]
bad_records = [ObservationRecord("red", "sensor-a", 7, 100)]
action_good, final_good, trace_good = run_belief_agent(belief, good_records, 2, 7, 100)
action_bad, _, trace_bad = run_belief_agent(belief, bad_records, 1, 7, 100)
assert action_good == "execute" and final_good[1] < DEFAULT_POLICY.danger_threshold
assert action_bad == "abort"
assert sum(event["state"] == "observe" for event in trace_good) == 2
assert all(event.get("status") == "accepted" for event in trace_good if event["state"] == "observe")


## 7. 陈旧 observation 与状态漂移：时间也是隐藏状态

网页/文件可能在观察后被他人修改。belief 要绑定 snapshot/version；高风险动作使用 optimistic concurrency，版本不一致就重新观察。仅保留摘要而丢版本会制造 TOCTOU。


In [ ]:
# stale/future/wrong-version 都被主 loop 消费预算但不更新 belief，也绝不能促成 execute。
record = ObservationRecord("green", "sensor-a", 7, 100)
assert version_matches(record, 7, 101, 2)
assert not version_matches(record, 8, 101, 2)
assert not version_matches(record, 7, 104, 2)
assert not version_matches(ObservationRecord("green", "sensor-a", 7, 102), 7, 101, 2)

safe_prior = np.array([0.95, 0.05])
no_cost_policy = POMDPArtifact("env-v5", "sensor-likelihood-v3", "evidence-gated-eig-v3", 0.08, 2, 0.0)
rejected_records = [
    ObservationRecord("green", "sensor-a", 7, 90),
    ObservationRecord("green", "sensor-a", 7, 102),
    ObservationRecord("green", "sensor-a", 8, 101),
]
expected_status = ["stale", "future", "wrong_version"]
for invalid_record, status in zip(rejected_records, expected_status):
    action, posterior, trace = run_belief_agent(safe_prior, [invalid_record], 1, 7, 101, no_cost_policy)
    observations = [event for event in trace if event["state"] == "observe"]
    assert action == "abort" and np.array_equal(posterior, safe_prior)
    assert [event["status"] for event in observations] == [status]
    assert not any(event.get("action") == "execute" for event in trace)

fresh_action, fresh_posterior, _ = run_belief_agent(safe_prior, [ObservationRecord("green", "sensor-a", 7, 101)], 1, 7, 101, no_cost_policy)
assert fresh_action == "execute" and fresh_posterior[1] < safe_prior[1]
for bad_stream, bad_budget in (([ObservationRecord("blue", "sensor-a", 7, 101)], 1), ([], -1), ([], 1.5)):
    try:
        run_belief_agent(belief, bad_stream, bad_budget, 7, 101, no_cost_policy); assert False
    except (TypeError, ValueError):
        assert True


## 8. 评测与制品：成功率之外必须报告损害与观察成本

比较任务成功、harm rate、unnecessary actions、inspect 次数、belief Brier/ECE、状态漂移恢复和 p95 延迟。环境 reset/snapshot、likelihood/阈值和动作 policy 都是可版本制品。


In [ ]:
def brier_danger(predicted, actual_danger):
    predicted, actual = np.asarray(predicted, dtype=float), np.asarray(actual_danger, dtype=float)
    if predicted.shape != actual.shape or not np.isfinite(predicted).all() or not np.isfinite(actual).all():
        raise ValueError("Brier 输入必须同形且有限")
    if ((predicted < 0) | (predicted > 1) | (actual < 0) | (actual > 1)).any():
        raise ValueError("Brier 输入必须位于 [0,1]")
    return float(np.mean((predicted - actual) ** 2))

def artifact_hash(artifact):
    validate_policy(artifact)
    return hashlib.sha256(json.dumps(asdict(artifact), sort_keys=True).encode()).hexdigest()

# 同一 observation 经 artifact 的 threshold/freshness 改变真实 loop 终态。
artifact = DEFAULT_POLICY
loose = POMDPArtifact("env-v5", "sensor-likelihood-v3", "evidence-gated-eig-v3", 0.15, 2, 0.0)
strict_threshold = POMDPArtifact("env-v5", "sensor-likelihood-v3", "evidence-gated-eig-v3", 0.08, 2, 0.0)
strict_freshness = POMDPArtifact("env-v5", "sensor-likelihood-v3", "evidence-gated-eig-v3", 0.15, 1, 0.0)
border_record = ObservationRecord("green", "sensor-a", 7, 8)
loose_action, _, loose_trace = run_belief_agent(belief, [border_record], 1, 7, 10, loose)
threshold_action, _, _ = run_belief_agent(belief, [border_record], 1, 7, 10, strict_threshold)
freshness_action, freshness_posterior, freshness_trace = run_belief_agent(belief, [border_record], 1, 7, 10, strict_freshness)
assert loose_action == "execute" and threshold_action == "abort" and freshness_action == "abort"
assert np.array_equal(freshness_posterior, belief)
assert any(event.get("status") == "stale" for event in freshness_trace)
digest = artifact_hash(artifact)
assert brier_danger([0, 1], [0, 1]) == 0 and len(digest) == 64
assert digest != artifact_hash(POMDPArtifact("env-v5", "sensor-v4", artifact.action_policy, artifact.danger_threshold, artifact.freshness_limit, artifact.inspect_cost))
assert any(event.get("fresh_evidence") for event in loose_trace if event["state"] == "decide")


## 面试收束：Agent/RAG 的算法只是控制面的一部分

推荐回答顺序是：任务目标和失败代价、状态/事件/证据合同、决策公式、可执行反例、离线与在线指标、权限和版本。受控环境只能证明状态机和数值关系，不能冒充开放网络、真实用户或真实模型结果。生产系统还要处理并发、超时、幂等、恶意内容、隐私、审计、灰度与回滚。

遇到追问时，主动区分模型判断与确定 verifier、计划与真实副作用、原始 observation 与 belief/memory、召回质量与生成归因，以及多尝试成功率与单次可靠性。
